# Edges, Contours & Features

> 📘 **Python Mastery** · Module 15 — Computer Vision · Lesson 3/5

Cleaning pixels (lesson 2) is preparation; this lesson is *understanding*: where objects end
(edges), what shape they have (contours), which points are structurally special (corners), and how
to find a known patch anywhere in an image (template matching). These classic techniques still
power OCR, industrial inspection and tracking systems — and they're the vocabulary every modern
detector assumes you know.

## 🎯 Learning Objectives

- **Describe** the four-stage edge pipeline: smooth → gradient → thinning → hysteresis thresholding
- **Tune** Canny's two thresholds and predict the effect of each
- **Extract** contours with `findContours` and overlay them using `drawContours`
- **Measure** contours: area, perimeter, centroid, bounding box and polygon approximation
- **Build** a working shape classifier from vertex counts (triangle vs square vs circle)
- **Detect** corners with Harris and `goodFeaturesToTrack`, and explain why corners matter
- **Locate** a sprite inside a larger scene with `matchTemplate` and `minMaxLoc`

## 1. What Exactly Is an Edge?

An edge is a place where brightness changes fast. Real detectors chain four steps:

1. **Smooth** — derivatives amplify noise, so blur first (Gaussian).
2. **Gradient** — measure change strength (`Sobel`) and its direction.
3. **Thinning** — gradients form thick ridges; keep only the ridge crest (one pixel wide).
4. **Hysteresis thresholding** — keep pixels above a *strong* threshold, plus any weaker pixels
   *connected* to them. This is the trick that turns broken fragment edges into continuous outlines.

`cv2.Canny` does all four in one call.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# A small "parts on a conveyor belt" inspection image
rng = np.random.default_rng(0)
noise = rng.normal(0, 6, (220, 340)).astype(np.int16)          # sensor noise
part = np.clip(np.full((220, 340), 90, dtype=np.int16) + noise, 0, 255).astype(np.uint8)
cv2.rectangle(part, (35, 45), (140, 175), 210, -1)              # bracket
cv2.circle(part, (250, 110), 62, 200, -1)                       # washer
cv2.circle(part, (250, 110), 22, 90, -1)                        # its hole

smoothed = cv2.GaussianBlur(part, (5, 5), 0)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].imshow(part, cmap="gray");           axes[0].set_title("Noisy parts")
axes[1].imshow(smoothed, cmap="gray");       axes[1].set_title("1) Gaussian smooth")
mag = cv2.magnitude(cv2.Sobel(smoothed, cv2.CV_64F, 1, 0, 3),
                    cv2.Sobel(smoothed, cv2.CV_64F, 0, 1, 3))
axes[2].imshow(mag, cmap="gray");            axes[2].set_title("2) Gradient magnitude")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Canny in One Line — and Its Two Thresholds

`edges = cv2.Canny(img, t1, t2)` runs the whole pipeline. The two thresholds implement the
hysteresis step: `t2` (high) defines *surely an edge*, `t1` (low) defines *maybe an edge* — weak
pixels survive only if connected through stronger ones. Practical ratio: **t2 ≈ 2–3 × t1**.

| Setting | Typical outcome |
|---------|-----------------|
| both thresholds too low | hairline noise everywhere |
| both too high | real boundaries vanish |
| good ratio (e.g. 60 / 180) | thin, connected, clean outlines |

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

page = np.full((280, 420), 235, dtype=np.uint8)          # our scanned-page fixture again
for i, line in enumerate(["Invoice #0042 - Dhaka", "3 x notebooks ..... 240 tk",
                          "1 x backpack ...... 950 tk", "TOTAL ............ 1250 tk"]):
    cv2.putText(page, line, (28, 50 + 38 * i), cv2.FONT_HERSHEY_SIMPLEX,
                0.55, (25, 25, 25), 2, cv2.LINE_AA)

triplets = [(20, 40, "too low: noise becomes edges"),
            (60, 180, "balanced ratio 1:3"),
            (150, 400, "too high: boundaries lost")]

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))
for ax, (t1, t2, ttl) in zip(axes, triplets):
    ax.imshow(cv2.Canny(page, t1, t2), cmap="gray")
    ax.set_title(f"Canny({t1}, {t2})\n{ttl}", fontsize=9)
    ax.axis("off")
plt.suptitle("Same page, three threshold choices")
plt.tight_layout()
plt.show()

## 3. Contours: `findContours` + `drawContours`

A **contour** is an ordered curve joining all continuous points of the same intensity along a
boundary — think of it as the object's outline stored as coordinates. `findContours` mutates nothing
but needs a *binary-ish* input (usually Otsu or Canny output); it returns a list of `(N, 1, 2)`
arrays plus a hierarchy table.

Retrieval modes control *which* curves you get:

| Mode | Gives you |
|------|-----------|
| `RETR_EXTERNAL` | outermost outlines only (most common) |
| `RETR_LIST` | all contours, no relationships |
| `RETR_TREE` | everything + parent/child links (holes!) |

**Syntax:**
```python
contours, hierarchy = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(canvas, contours, contourIdx=-1, color=(0, 255, 0), thickness=2)
```

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

part = np.full((220, 340), 90, dtype=np.uint8)
cv2.rectangle(part, (35, 45), (140, 175), 210, -1)     # solid bracket...
cv2.rectangle(part, (70, 80), (105, 130), 90, -1)      # ...with a hole inside
cv2.circle(part, (255, 110), 58, 205, -1)

_, bw = cv2.threshold(part, 127, 255, cv2.THRESH_BINARY)
cnts_ext, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cnts_all, hier = cv2.findContours(bw, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

print("EXTERNAL found:", len(cnts_ext), "contour(s)")
print("TREE found    :", len(cnts_all), "contours (outer + inner hole)")
print("hierarchy shape:", None if hier is None else hier.shape,
      "-> rows = [Next, Previous, First_Child, Parent]")

overlay = cv2.cvtColor(part, cv2.COLOR_GRAY2BGR)
cv2.drawContours(overlay, cnts_ext, -1, (0, 255, 0), 3)          # -1 = draw ALL
cv2.drawContours(overlay, cnts_all, 1, (0, 0, 255), 2)           # the hole, in red

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].imshow(bw, cmap="gray");               axes[0].set_title("Binary input")
axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); axes[1].set_title("Green: external, red: hole contour")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Contour array format:", cnts_ext[0].shape, "| first points:", cnts_ext[0][:3].ravel())

## 4. Contour Properties You'll Use Constantly

Once you have a contour, geometry comes free: `contourArea`, `arcLength` (perimeter), image moments
(centroid), `boundingRect` (axis-aligned box) and `approxPolyDP` (collapse the point cloud into a
few vertices — the key to recognizing shapes).

**Syntax:**
```python
area = cv2.contourArea(c)
peri = cv2.arcLength(c, True)                    # True = closed curve
M = cv2.moments(c); cx = int(M["m10"] / M["m00"])
x, y, w, h = cv2.boundingRect(c)
approx = cv2.approxPolyDP(c, eps, True)          # eps ~ 2% of perimeter
```

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

canvas = np.zeros((240, 340), dtype=np.uint8)
cv2.rectangle(canvas, (30, 50), (130, 170), 255, -1)      # rectangle
tri = np.array([(190, 40), (320, 100), (215, 190)], np.int32)
cv2.fillPoly(canvas, [tri], 255)                          # triangle

cnts, _ = cv2.findContours(canvas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
info = cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)

for c in cnts:
    area = cv2.contourArea(c)
    peri = cv2.arcLength(c, True)
    M = cv2.moments(c)
    cx, cy = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
    x, y, w, h = cv2.boundingRect(c)
    approx = cv2.approxPolyDP(c, 0.02 * peri, True)
    kind = {3: "triangle", 4: "rectangle"}.get(len(approx), f"{len(approx)}-gon")
    print(f"{kind:9s} area={area:7.0f}  perimeter={peri:6.1f}  centroid=({cx},{cy})  bbox={w}x{h}")
    cv2.rectangle(info, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.drawContours(info, [approx], -1, (0, 0, 255), 2)
    cv2.circle(info, (cx, cy), 4, (255, 0, 255), -1)

plt.figure(figsize=(6, 4))
plt.imshow(cv2.cvtColor(info, cv2.COLOR_BGR2RGB))
plt.title("Green bbox | red approxPolygon | magenta centroid")
plt.axis("off")
plt.show()

## 5. Mini-Project: The Shape Classifier Game

Time to cash in: a complete, working classifier in ~20 lines. Strategy —

1. draw shapes we can't "see" (programmatic, shuffled order),
2. binarize → find external contours,
3. count `approxPolyDP` vertices: **3 → triangle, 4 → quadrilateral, >6 → circle**
   (circles need many segments to approximate).

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

def draw_shape(kind, size=160):
    img = np.zeros((size, size), dtype=np.uint8)
    cxy = size // 2
    if kind == "circle":
        cv2.circle(img, (cxy, cxy), size // 2 - 12, 255, -1)
    else:                                            # square + regular polygons
        n = {"square": 4, "triangle": 3, "pentagon": 5}[kind]
        ang = np.linspace(0, 2 * np.pi, n, endpoint=False) - np.pi / 2
        if kind != "square":
            ang -= np.pi / n                          # sit flat like a real square
        r = size // 2 - 12
        pts = np.column_stack([cxy + r * np.cos(ang), cxy + r * np.sin(ang)]).astype(np.int32)
        cv2.fillPoly(img, [pts], 255)
    return img

def classify(verts):                                  # the whole "model"
    return {3: "triangle", 4: "square", 5: "pentagon"}.get(verts,
           "circle" if verts > 6 else f"{verts}-gon")

truth = ["triangle", "square", "pentagon", "circle"]
order = np.random.default_rng(42).permutation(truth)          # shuffle so we can't peek

panels, predictions = [], []
for name in order:
    img = draw_shape(name)
    cnts, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = max(cnts, key=cv2.contourArea)                        # biggest blob wins
    verts = len(cv2.approxPolyDP(c, 0.02 * cv2.arcLength(c, True), True))
    guess = classify(verts)
    predictions.append((name, guess, verts))
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(vis, [c], -1, (0, 255, 0), 2)
    panels.append((vis, name))

score = sum(t == g for t, g, _ in predictions)
for t, g, v in predictions:
    print(f"true={t:9s} vertices={v:2d} -> predicted: {g}")
print(f"\nClassifier score: {score}/{len(predictions)} correct")

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, (vis, ttl) in zip(axes, panels):
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax.set_title(f"drew '{ttl}'"); ax.axis("off")
plt.tight_layout()
plt.show()

> 🔍 **Under the Hood:** `approxPolyDP` is the **Douglas–Peucker algorithm**: it keeps the two most
> distant points, finds the point farthest from that segment, and recursively splits wherever the
> deviation exceeds `eps`. That's why a perfect circle at `eps = 2%` still yields 10–20 vertices while
> a square snaps to exactly 4 — vertex count is really measuring *"how many straight runs the outline
> has"*, which is precisely what distinguishes a circle from a polygon. Also note `contourArea` uses
> Green's theorem (shoelace formula), so it's exact for non-self-intersecting curves and far faster
> than counting pixels.

## 6. Corners: Harris & `goodFeaturesToTrack`

Edges tell you where a boundary runs; **corners** tell you where boundaries *turn* — and unlike
edges, a corner looks different under almost any shift, which makes corners ideal anchors for
tracking, panorama stitching and camera calibration.

Harris scores each window by how much the intensity changes when the window slides in *any*
direction. `goodFeaturesToTrack` (Shi–Tomasi refinement) then picks the strongest, well-spread
points — the standard input for optical-flow trackers like `cv2.calcOpticalFlowPyrLK`.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

board = np.kron([[1, 0] * 5, [0, 1] * 5] * 5, np.ones((40, 40))).astype(np.float32) * 255
gray = board.astype(np.uint8)                                   # checkerboard: corner-rich

harris = cv2.cornerHarris(np.float32(gray), blockSize=2, ksize=3, k=0.04)
harris_d = cv2.dilate(harris, None)                             # fatten maxima for visibility
hits = harris_d > 0.05 * harris_d.max()

corners = cv2.goodFeaturesToTrack(np.float32(gray), maxCorners=25,
                                  qualityLevel=0.05, minDistance=10)
overlay2 = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
for pt in corners.reshape(-1, 2):
    x, y = int(pt[0]), int(pt[1])
    cv2.circle(overlay2, (x, y), 7, (0, 255, 0), 2)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(gray, cmap="gray");                        axes[0].set_title("Checkerboard")
im1 = axes[1].imshow(harris, cmap="inferno");             axes[1].set_title("Raw cornerHarris response")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
axes[2].imshow(cv2.cvtColor(overlay2, cv2.COLOR_BGR2RGB)); axes[2].set_title(
    f"goodFeaturesToTrack: top {len(corners)} points (green)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Harris flagged", int(hits.sum()), "pixels;", len(corners), "survived quality+distance filtering")

## 7. Template Matching: Find This Patch in That Scene

Give `matchTemplate` a small **template** and a bigger **scene**; it slides the template over every
position and scores similarity into a result map (here: normalized cross-correlation, `TM_CCOEFF_NORMED`,
range −1…1). With `TM_SQDIFF` you minimize instead; with `TM_CCOEFF_NORMED` you maximize.
`minMaxLoc` reads out the winner. Limitations to remember: no scale or rotation invariance —
the template must appear at the same size and angle.

| Real-world use | What's matched |
|----------------|----------------|
| Document OCR pre-pass | digit/letter tiles |
| Industrial QA | missing component on a PCB |
| Game bots / UI testing | button icons on screen |
| Medical imaging | vertebra templates in X-rays |

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)
scene_f = rng.random((260, 380)) * 60                          # busy low-gray background
scene = cv2.GaussianBlur(scene_f.astype(np.uint8), (3, 3), 0)

sprite = np.zeros((36, 48), dtype=np.uint8)                    # the little "arrow tag"
pts = np.array([[4, 18], [28, 4], [28, 12], [44, 12], [44, 24], [28, 24], [28, 32]], np.int32)
cv2.fillPoly(sprite, [pts], 255)

positions = [(60, 40), (230, 150)]                             # hide TWO copies
for x, y in positions:
    scene[y:y + 36, x:x + 48] = sprite

res = cv2.matchTemplate(scene, sprite, cv2.TM_CCOEFF_NORMED)
mn, mx, mnloc, mxloc = cv2.minMaxLoc(res)
print(f"Best match score {mx:.3f} at (x={mxloc[0]}, y={mxloc[1]})")

found = cv2.cvtColor(scene, cv2.COLOR_GRAY2BGR)
h, w = sprite.shape
ys, xs = np.where(res >= 0.85)                                 # collect ALL strong hits
boxes = []
for x, y in zip(xs, ys):
    if all(abs(x - bx) > w // 2 or abs(y - by) > h // 2 for bx, by in boxes):
        boxes.append((x, y))                                   # skip duplicates of same spot
print(f"{len(boxes)} distinct location(s) above 0.85:")
for x, y in boxes:
    cv2.rectangle(found, (x, y), (x + w, y + h), (0, 0, 255), 2)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].imshow(sprite, cmap="gray");                axes[0].set_title("Template (sprite)")
im1 = axes[1].imshow(res, cmap="viridis");          axes[1].set_title("Score map res")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
axes[2].imshow(cv2.cvtColor(found, cv2.COLOR_BGR2RGB)); axes[2].set_title("Detections drawn in red")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Detected positions:", boxes, "| true positions:", positions)

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---------|---------|-----|
| Unpacking `_, cnts, hier = findContours(...)` copied from old tutorials | OpenCV ≥ 4 returns **two** values → ValueError | `cnts, hier = cv2.findContours(...)` |
| Running `findContours` on the raw grayscale photo | It treats every value band as structure; garbage curves | Threshold or Canny first |
| Forgetting `contourIdx=-1` in `drawContours` | Only draws... actually default `-1` draws all; passing `0` draws just one | Pass `-1` for all, index for one |
| Treating the template-match score as a probability | `TM_CCOEFF_NORMED` is a correlation, not calibrated confidence | Validate with known negatives; tune cutoff empirically |
| Template matching across zoom levels | No scale invariance — score collapses | Resize templates over scales, or use feature matching (ORB/SIFT) |
| Sorting contours by list order | Order follows scan position, not importance | Sort explicitly: `sorted(cnts, key=cv2.contourArea, reverse=True)` |

## 💡 Best Practices & Pro Tips

- **Chain the trio deliberately**: blur → Canny → findContours covers most "find the objects" tasks;
  keep the intermediate maps until results look right.
- **Filter by area before analyzing**: after `findContours`, drop blobs below a minimum
  `contourArea` — specks die there instead of polluting your statistics.
- **Use `CHAIN_APPROX_SIMPLE`** to compress collinear contour points (memory-friendly); use
  `CHAIN_APPROX_NONE` when you need every boundary pixel.
- **Corners for tracking, templates for exact appearance.** If the target may rotate/scale, jump to
  feature matching (ORB) — template matching will betray you silently, not loudly.
- 🤖 **AI-engineering relevance:** these functions generate labels and crop training data — sliding
  windows produce candidate patches, contour bounding boxes become YOLO annotations, and
  `goodFeaturesToTrack` seeds tracking datasets. Classic CV is still the data-labeling workhorse of
  modern deep learning pipelines.

## 📌 Summary

| Method | What it does | Example |
|--------|--------------|---------|
| `cv2.Canny(img, t1, t2)` | Full edge pipeline, hysteresis thresholds | `cv2.Canny(gray, 60, 180)` |
| `cv2.findContours(bw, mode, method)` | Ordered boundary curves | `RETR_EXTERNAL, CHAIN_APPROX_SIMPLE` |
| `cv2.drawContours(img, cs, -1, col, th)` | Overlay outlines (-1 = all) | green outline width 2 |
| `cv2.contourArea / arcLength` | Size / perimeter | sort & filter blobs |
| `cv2.moments(c)` → centroid | Center of mass | `m10/m00, m01/m00` |
| `cv2.boundingRect(c)` | Axis-aligned box | crop object patches |
| `cv2.approxPolyDP(c, eps, True)` | Vertex-count shape signature | `eps = 0.02 * perimeter` |
| `cv2.cornerHarris / goodFeaturesToTrack` | Corner anchors for tracking | `maxCorners=25, minDistance=10` |
| `cv2.matchTemplate(scene, tpl, TM_CCOEFF_NORMED)` | Slide-and-score localization | + `cv2.minMaxLoc(res)` |

- Edges = strong brightness change; Canny = smooth → gradient → thin → connect.
- Contours turn binary blobs into computable geometry (area, box, vertices).
- Vertex count from `approxPolyDP` is enough for a working shape classifier.
- Corners are stable anchors; template matching finds exact-size copies — know each tool's limits.

## 🔗 Next Lesson

Continue to **[04_CNN_Image_Classification](../04_CNN_Image_Classification/notes.ipynb)** — hand the
feature engineering to the machine: convolutional neural networks for image classification.